In [ ]:
from dotenv import load_dotenv
from pathlib import Path, PurePath, __file__
import os
import logging
from joblib import dump, load
from typing import Tuple, List, Dict, Any
import pandas as pd
import nflreadpy as nfl
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.ensemble import (RandomForestRegressor,HistGradientBoostingRegressor)
from sklearn.metrics import mean_squared_error, r2_score


# Locations
BACKEND_DIR = Path(__file__).parent
BASE_DIR = BACKEND_DIR.parent
DATA_DIR = BACKEND_DIR / "data"
MODELS_DIR = BACKEND_DIR / "models"
LOG_DIR = BACKEND_DIR / "logs"
FRONTEND_DIR = BASE_DIR / "frontend"
FRONTEND_DIST = FRONTEND_DIR / "dist"
FRONTEND_BUILD = FRONTEND_DIST  # Alias for compatibility

# Truthy parsing helper
TRUTHY = {"true", "t", "1", "yes", "y"}

def _load_env() -> None:
    """
    Load .env from backend or repo root.
    """
    dotenv_loaded = load_dotenv(BACKEND_DIR / ".env")

    if not dotenv_loaded:
        load_dotenv(BASE_DIR / ".env")
        print("Loaded .env from repo root")
        return load_dotenv(BASE_DIR / ".env")
    print(f"Loaded .env from backend: {dotenv_loaded}")
    return dotenv_loaded

print(f"Loading .env...")
_load_env()
print(f".env loaded.{os.listdir(BACKEND_DIR)}")

In [ ]:
import math
from build_csv_datasetsv3 import build_dataset
from backend.utils import   

# ---------------------------------------------------------------------
# Load Models from disk
# ---------------------------------------------------------------------

clf_path = MODELS_DIR / "histgradient_home_win_clf.joblib"
log = logging.getLogger("backend.main")

def _predict_home_win_prob():
    """Predict home win probability using histgradient classifier with fallback to logistic function."""
    clf = load(clf_path)
    df = build_dataset(2023, 2023, DATA_DIR / "predictions_temp")


    # Try direct prediction
    if hasattr(clf, "predict_proba"):
        try:
            proba = clf.predict_proba(X_raw)
            idx = _pick_positive_class_index(clf)
            p = float(proba[0][idx])
            return float(np.clip(p, 0.0, 1.0)), False
        except Exception as e:
            log.warning("[Predict] hist_win_clf predict_proba(raw) failed: %s", e)

        # Try transformed prediction
        try:
            X_tx = bundle.preprocessor.transform(_safe_fill(X_raw))
            proba = clf.predict_proba(X_tx)
            idx = _pick_positive_class_index(clf)
            p = float(proba[0][idx])
            return float(np.clip(p, 0.0, 1.0)), False
        except Exception as e:
            log.warning("[Predict] hist_win_clf predict_proba(preprocessed) failed: %s", e)

    # Fallback to logistic function
    p = 1.0 / (1.0 + math.exp(-0.25 * float(point_diff)))
    return float(np.clip(p, 0.0, 1.0)), True


In [ ]:
import pandas as pd
import os
import requests
from pydantic import BaseModel
from typing import Dict, Any

class TeamCard(BaseModel):
    key: str
    value: Any
   


logo_df = pd.read_csv("../backend/data/team_logos.csv")
print(logo_df.head())

for key, value in logo_df.iterrows():
  print(key, value)
  team_card = TeamCard(key=str(key), value=value)
  print(team_card)  

In [5]:
import pandas as pd

df = pd.read_csv("./data/datasets/game_features_20260109.csv")
print(df.info())
df = df.convert_dtypes(infer_objects=True)
print(df.info())
df = df.sort_values(by=["season","week"], ascending=[False, False]).reset_index(drop=True)
print(df.head(10))
df.to_csv("./data/datasets/game_features_20260109.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2755 entries, 0 to 2754
Columns: 247 entries, season to away_team_WAS
dtypes: bool(68), float64(164), int64(4), object(11)
memory usage: 3.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2755 entries, 0 to 2754
Columns: 247 entries, season to away_team_WAS
dtypes: Float64(129), Int64(39), boolean(69), string(10)
memory usage: 4.5 MB
None
   season  week          game_id home_game_date home_team away_team  \
0    2025    19  2025_19_BUF_JAX     2026-01-11       JAX       BUF   
1    2025    19   2025_19_GB_CHI     2026-01-10       CHI        GB   
2    2025    19  2025_19_HOU_PIT     2026-01-12       PIT       HOU   
3    2025    19   2025_19_LAC_NE     2026-01-11        NE       LAC   
4    2025    19  2025_19_LAR_CAR     2026-01-10       CAR       LAR   
5    2025    19   2025_19_LA_CAR     2026-01-10       CAR       LAR   
6    2025    19   2025_19_SF_PHI     2026-01-11       PHI        SF   
7    2025    18   2025_18_ARI